In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from matplotlib.lines import Line2D

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'

dates = pd.date_range("2024-07-01", periods=168, freq="h")
hours = np.arange(168)

def daily_pattern(h):
    t = (h % 24) / 24
    return (40 + 18*np.sin(np.pi*(t-0.05))
               + 10*np.exp(-((t-0.83)**2)/0.01)
               - 6*np.exp(-((t-0.15)**2)/0.008))

weekly_scale = np.array([1.0, 1.02, 1.03, 1.02, 1.04, 0.92, 0.88])
weekly_mult  = np.array([weekly_scale[int(h // 24)] for h in hours])
np.random.seed(7)
true_demand  = daily_pattern(hours) * weekly_mult + np.random.normal(0, 0.5, 168)

fcd_starts = [0, 24, 48, 72, 96, 120]
colors_fc  = ["#3B4CC0", "#7B3FA0", "#B0308A", "#D4622A", "#E8A020", "#F5D000"]
SHADE      = "#C0675A"
QUANTILES  = np.array([0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9])

def pinball(q, y, f):
    e = y - f
    return np.where(e >= 0, q*e, (q-1)*e).mean()

def compute_scrps(fc, true_d):
    seg = true_d[fc["start"]:fc["end"]]
    qf  = {0.1:fc["q10"], 0.25:fc["q25"], 0.4:fc["q40"], 0.5:fc["median"],
           0.6:fc["q60"], 0.75:fc["q75"], 0.9:fc["q90"]}
    return np.mean([pinball(q, seg, qf[q]) for q in QUANTILES])

def compute_mae(fc, true_d):
    return np.abs(fc["median"] - true_d[fc["start"]:fc["end"]]).mean()

def mean_abs_revision(forecasts):
    revisions = []
    for i in range(len(forecasts)-1):
        a, b = forecasts[i], forecasts[i+1]
        ov_s, ov_e = b["start"], a["end"]
        if ov_s >= ov_e: continue
        revisions.append(np.abs(
            a["median"][ov_s-a["start"]:ov_e-a["start"]] -
            b["median"][0:ov_e-b["start"]]
        ).mean())
    return np.mean(revisions)

def fc_dict(median, s10, wobble, start, end):
    s25, s40 = s10*0.55, s10*0.25
    return dict(median=median,
                q10=median-s10+wobble, q90=median+s10+wobble,
                q25=median-s25,        q75=median+s25,
                q40=median-s40,        q60=median+s40,
                start=start, end=end)

# ══════════════════════════════════════════════════════════════════════════
# SLIDE 1 — Same MAE, different band width → motivates sCRPS
# ══════════════════════════════════════════════════════════════════════════
np.random.seed(42)
slide1_tight, slide1_wide = [], []
for start in fcd_starts:
    end = min(start+48, 168)
    seg = true_demand[start:end].copy()
    n   = len(seg)
    noise  = np.random.normal(0, 1.1, n)
    median = seg + noise
    slide1_tight.append(fc_dict(median, np.linspace(2.8, 1.8, n), np.zeros(n), start, end))
    wide_s10 = np.abs(np.random.normal(0, 5, n)) + 9
    slide1_wide.append(fc_dict(median, wide_s10, np.random.normal(0, 2, n), start, end))

mae_t1  = np.mean([compute_mae(fc,  true_demand) for fc in slide1_tight])
mae_w1  = np.mean([compute_mae(fc,  true_demand) for fc in slide1_wide])
crps_t1 = np.mean([compute_scrps(fc, true_demand) for fc in slide1_tight])
crps_w1 = np.mean([compute_scrps(fc, true_demand) for fc in slide1_wide])
print(f"\nSlide 1 — Same MAE, different sCRPS")
print(f"  Tight:  MAE={mae_t1:.2f}  sCRPS={crps_t1:.2f}")
print(f"  Wide:   MAE={mae_w1:.2f}  sCRPS={crps_w1:.2f}")

# ══════════════════════════════════════════════════════════════════════════
# SLIDE 2 — Same MAE + sCRPS, different stability → motivates stability metric
#
# KEY IDEA:
#   STABLE:  shared global noise (sigma_g) + small per-FCD jitter (eps)
#            → FCD lines track together but are visibly distinct
#            → MAE driven by global noise magnitude
#
#   ERRATIC: independent per-FCD noise (sigma_e) + alternating shift on
#            the overlap region only (second half of each window)
#            → FCD lines diverge in overlap, stable in first half
#            → MAE matched to stable by construction (grid-searched)
#
#   Both use same band width → sCRPS nearly identical
# ══════════════════════════════════════════════════════════════════════════
SIGMA_G = 3.0   # global noise std for stable
EPS     = 0.8   # per-FCD jitter for stable (keeps lines distinct but close)
SIGMA_E = 2.0   # per-FCD noise std for erratic
AMP     = 3.0   # overlap shift amplitude for erratic
BAND    = 3.5   # quantile band half-width (same for both)

np.random.seed(42)
global_noise = np.random.normal(0, SIGMA_G, 168)

def make_slide2_stable():
    np.random.seed(13)
    results = []
    for start in fcd_starts:
        end = min(start + 48, 168)
        n   = end - start
        jitter = np.random.normal(0, EPS, n)
        median = true_demand[start:end] + global_noise[start:end] + jitter
        results.append(fc_dict(median, np.full(n, BAND), np.zeros(n), start, end))
    return results

def make_slide2_erratic():
    np.random.seed(77)
    results = []
    for i, start in enumerate(fcd_starts):
        end = min(start + 48, 168)
        n   = end - start
        noise = np.random.normal(0, SIGMA_E, n)
        shift = np.zeros(n)
        shift[n//2:] = (-1)**i * AMP   # alternating shift on overlap region only
        median = true_demand[start:end] + noise + shift
        results.append(fc_dict(median, np.full(n, BAND), np.zeros(n), start, end))
    return results

slide2_erratic = make_slide2_erratic()
slide2_stable  = make_slide2_stable()

mae_e2  = np.mean([compute_mae(fc,  true_demand) for fc in slide2_erratic])
mae_s2  = np.mean([compute_mae(fc,  true_demand) for fc in slide2_stable])
crps_e2 = np.mean([compute_scrps(fc, true_demand) for fc in slide2_erratic])
crps_s2 = np.mean([compute_scrps(fc, true_demand) for fc in slide2_stable])
rev_e2  = mean_abs_revision(slide2_erratic)
rev_s2  = mean_abs_revision(slide2_stable)
print(f"\nSlide 2 — Same MAE + sCRPS, different stability")
print(f"  Erratic: MAE={mae_e2:.2f}  sCRPS={crps_e2:.2f}  Revision={rev_e2:.2f}")
print(f"  Stable:  MAE={mae_s2:.2f}  sCRPS={crps_s2:.2f}  Revision={rev_s2:.2f}")

# ══════════════════════════════════════════════════════════════════════════
# Draw
# ══════════════════════════════════════════════════════════════════════════
def draw(forecasts, outpath, subtitle):
    fontsize = 22
    fig, ax  = plt.subplots(figsize=(20, 4.5))
    ax.plot(dates, true_demand, color="black", linewidth=1.2, zorder=6)
    for fc in forecasts:
        s, e = fc["start"], fc["end"]
        t = dates[s:e]
        ax.fill_between(t, fc["q10"], fc["q90"], color=SHADE, alpha=0.10, linewidth=0)
        ax.fill_between(t, fc["q25"], fc["q75"], color=SHADE, alpha=0.13, linewidth=0)
        ax.fill_between(t, fc["q40"], fc["q60"], color=SHADE, alpha=0.18, linewidth=0)
    for i, fc in enumerate(forecasts):
        s, e = fc["start"], fc["end"]
        ax.plot(dates[s:e], fc["median"], color=colors_fc[i],
                linewidth=1.5, alpha=0.92, zorder=4)
        ax.axvline(dates[s], color=colors_fc[i],
                   linewidth=0.7, linestyle=":", alpha=0.35, zorder=3)
    ax.set_ylabel("Demand (GW)", fontsize=fontsize)
    ax.set_ylim(10, 95); ax.set_yticks([20,30,40,50,60,70,80])
    ax.set_xlabel("Date", fontsize=fontsize)
    ax.text(0.0, 1.02, subtitle, transform=ax.transAxes,
            fontsize=fontsize+4, color="#444444", fontstyle="italic", va="bottom")
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%a\n%-d %b"))
    ax.xaxis.set_minor_locator(mdates.HourLocator(byhour=[6,12,18]))
    plt.xticks(fontsize=fontsize); plt.yticks(fontsize=fontsize)
    legend_handles = [
        Line2D([0],[0], color="black",       linewidth=2.0, label="Ground Truth"),
        Line2D([0],[0], color=colors_fc[0],  linewidth=1.7, label="First FCD Forecast"),
        Line2D([0],[0], color=colors_fc[-1], linewidth=1.7, label="Last FCD Forecast"),
        plt.matplotlib.patches.Patch(facecolor=SHADE, alpha=0.4,
                                     label="Quantile bands\n(10/25/40–60/75/90%)"),
    ]
    ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(1.01,1.0),
              borderaxespad=0, fontsize=fontsize-4, frameon=True, framealpha=0.9)
    fig.tight_layout()
    fig.savefig(outpath, dpi=160, bbox_inches="tight")
    #plt.close(fig)

# Slide 1
draw(slide1_wide,
     "/home/wpotosna/fcst_accuracy_wide_bands.pdf",
     f"MAE = {mae_w1:.1f} GW   |   sCRPS = {crps_w1:.1f} GW")
draw(slide1_tight,
     "/home/wpotosna/fcst_accuracy_tight_bands.pdf",
     f"MAE = {mae_t1:.1f} GW   |   sCRPS = {crps_t1:.1f} GW")

# Slide 2
draw(slide2_erratic,
     "/home/wpotosna/fcst_stability_erratic.pdf",
     f"MAE = {mae_e2:.1f} GW   |   sCRPS = {crps_e2:.1f} GW")
draw(slide2_stable,
     "/home/wpotosna/fcst_stability_stable.pdf",
     f"MAE = {mae_s2:.1f} GW   |   sCRPS = {crps_s2:.1f} GW")

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'

dates = pd.date_range("2024-07-01", periods=168, freq="h")
hours = np.arange(168)

def daily_pattern(h):
    t = (h % 24) / 24
    return (40 + 18 * np.sin(np.pi * (t - 0.05))
               + 10 * np.exp(-((t - 0.83) ** 2) / 0.01)
               - 6  * np.exp(-((t - 0.15) ** 2) / 0.008))

weekly_scale = np.array([1.0, 1.02, 1.03, 1.02, 1.04, 0.92, 0.88])
weekly_mult  = np.array([weekly_scale[int(h // 24)] for h in hours])
np.random.seed(7)
true_demand  = daily_pattern(hours) * weekly_mult + np.random.normal(0, 0.5, 168)

fcd_starts = [0, 24, 48, 72, 96, 120]
colors_fc  = ["#3B4CC0", "#7B3FA0", "#B0308A", "#D4622A", "#E8A020", "#F5D000"]
SHADE      = "#C0675A"
BAND       = 5.0
global_mean = true_demand.mean()

fontsize = 22
fig, ax  = plt.subplots(figsize=(20, 4.5))

ax.plot(dates, true_demand, color="black", linewidth=1.2, zorder=6)

for i, start in enumerate(fcd_starts):
    end = min(start + 48, 168)
    t   = dates[start:end]
    med = np.full(end - start, global_mean)
    ax.fill_between(t, med - BAND, med + BAND,
                    color=SHADE, alpha=0.12, linewidth=0)
    ax.plot(t, med, color=colors_fc[i], linewidth=1.5, alpha=0.92, zorder=4)
    ax.axvline(dates[start], color=colors_fc[i],
               linewidth=0.7, linestyle=":", alpha=0.35, zorder=3)

ax.set_ylabel("Demand (GW)", fontsize=fontsize)
ax.set_ylim(10, 95)
ax.set_yticks([20, 30, 40, 50, 60, 70, 80])
ax.set_xlabel("Date", fontsize=fontsize)
ax.text(0.0, 1.02,
        "Mean prediction — all FCDs identical  |  sFPC = 0.00  |  sEV = high",
        transform=ax.transAxes, fontsize=fontsize + 4,
        color="#444444", fontstyle="italic", va="bottom")

ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a\n%-d %b"))
ax.xaxis.set_minor_locator(mdates.HourLocator(byhour=[6, 12, 18]))
plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)

legend_handles = [
    Line2D([0],[0], color="black",       linewidth=2.0, label="Ground Truth"),
    Line2D([0],[0], color=colors_fc[0],  linewidth=1.7, label="First FCD Forecast"),
    Line2D([0],[0], color=colors_fc[-1], linewidth=1.7, label="Last FCD Forecast"),
    plt.matplotlib.patches.Patch(facecolor=SHADE, alpha=0.4, label="Quantile bands"),
]
ax.legend(handles=legend_handles, loc="upper left",
          bbox_to_anchor=(1.01, 1.0), borderaxespad=0,
          fontsize=fontsize - 4, frameon=True, framealpha=0.9)

fig.tight_layout()
#plt.savefig("slide3_mean_prediction.png", dpi=160, bbox_inches="tight")
plt.show()
print("Done.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation, PillowWriter

rng = np.random.default_rng(0)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'

C_LINE  = "#2C4A7C"
C_CTX   = "#5B7DB5"
C_HOR   = "#CA6F6A"
C_FDATE = "#E67E22"
C_GRAY  = "#9CA3AF"

T = 80
t = np.arange(T)
series = (
    1.5
    + 0.6 * np.sin(2 * np.pi * t / 30)
    + 0.3 * np.sin(2 * np.pi * t / 12)
    + rng.normal(0, 0.15, T)
)

CONTEXT   = 20
HORIZON   = 12
MAX_START = T - CONTEXT - HORIZON

window_starts = [2, 12, 20, 35, 45, MAX_START - 2]
HERO_IDX  = 2
HOLD      = 140   # hold each window for one full FS sweep (148 frames = 5.9s)
TRANS     = 8
PAUSE     = 18

frames = []
for i, ws in enumerate(window_starts):
    if i == 0:
        frames += [float(ws)] * HOLD
    else:
        prev = window_starts[i - 1]
        for f in range(TRANS):
            a = f / (TRANS - 1)
            a = a * a * (3 - 2 * a)
            frames.append(prev + (ws - prev) * a)
        hold = HOLD + (PAUSE if i == HERO_IDX else 0)
        frames += [float(ws)] * hold

HERO_WS = float(window_starts[HERO_IDX])

fig, ax = plt.subplots(figsize=(10, 3.8))
fig.subplots_adjust(left=0.04, right=0.97, top=0.88, bottom=0.22)

y_min = series.min() - 0.55
y_max = series.max() + 0.82
ax.set_xlim(-1, T)
ax.set_ylim(y_min, y_max)
ax.set_xlabel("Time  t", fontsize=16)
ax.set_ylabel("y", fontsize=16, rotation=0, labelpad=16)
ax.tick_params(left=False, labelleft=False, labelsize=16)

ax.plot(t, series, color=C_LINE, linewidth=1.8, zorder=3)
ax.scatter(t, series, color=C_LINE, s=18, zorder=4)

rect_ctx = patches.Rectangle((0, y_min), CONTEXT, y_max - y_min,
                               color=C_CTX, alpha=0.18, zorder=1)
rect_hor = patches.Rectangle((CONTEXT, y_min), HORIZON, y_max - y_min,
                               color=C_HOR, alpha=0.28, zorder=1)
ax.add_patch(rect_ctx)
ax.add_patch(rect_hor)

vl = ax.axvline(0,               color=C_CTX,  lw=1.1, ls="--", alpha=0.8)
vm = ax.axvline(CONTEXT,         color=C_GRAY, lw=1.0, ls="--", alpha=0.7)
vr = ax.axvline(CONTEXT+HORIZON, color=C_HOR,  lw=1.1, ls="--", alpha=0.8)

dot_fcd, = ax.plot([], [], "o", color=C_FDATE, ms=11,
                   mec="white", mew=1.4, zorder=6)

brk_y  = y_max - 0.08
brk_yt = y_max + 0.02

arr_ctx = ax.annotate("", xy=(CONTEXT, brk_y), xytext=(0, brk_y),
                       arrowprops=dict(arrowstyle="<->", color=C_CTX, lw=1.3))
arr_hor = ax.annotate("", xy=(CONTEXT+HORIZON, brk_y), xytext=(CONTEXT, brk_y),
                       arrowprops=dict(arrowstyle="<->", color=C_HOR, lw=1.3))

lbl_ctx = ax.text(CONTEXT/2, brk_yt, "context  L",
                  ha="center", va="bottom", fontsize=14,
                  color=C_CTX, fontweight="bold")
lbl_hor = ax.text(CONTEXT+HORIZON/2, brk_yt, "horizon  H",
                  ha="center", va="bottom", fontsize=14,
                  color=C_HOR, fontweight="bold")

lbl_fcd = ax.text(0, 0, "FCD", ha="left", va="top",
                  fontsize=14, color=C_FDATE, fontweight="bold")


def update(fi):
    ws = frames[fi]
    we = ws + CONTEXT
    wh = we + HORIZON

    rect_ctx.set_x(ws)
    rect_hor.set_x(we)
    vl.set_xdata([ws, ws])
    vm.set_xdata([we, we])
    vr.set_xdata([wh, wh])

    idx = max(0, min(int(round(we)), T - 1))
    dot_fcd.set_data([we], [series[idx]])

    arr_ctx.xy    = (we, brk_y);  arr_ctx.xyann = (ws, brk_y)
    arr_hor.xy    = (wh, brk_y);  arr_hor.xyann = (we, brk_y)
    lbl_ctx.set_x(ws + CONTEXT / 2)
    lbl_hor.set_x(we + HORIZON / 2)

    lbl_fcd.set_text(f"FCD  t={idx}")
    lbl_fcd.set_visible(True)
    lbl_fcd.set_ha("left")
    lbl_fcd.set_position((we + 0.6, series[idx] - 0.15))

    return (rect_ctx, rect_hor, vl, vm, vr, dot_fcd,
            arr_ctx, arr_hor, lbl_ctx, lbl_hor, lbl_fcd)


ani = FuncAnimation(fig, update, frames=len(frames), interval=40, blit=False)
ani.save("window_sampling.gif", writer=PillowWriter(fps=25), dpi=130)
print(f"✓ saved  ({len(frames)} frames = {len(frames)/25:.1f}s)")

In [ ]:
import numpy as np
import matplotlib.patches as patches
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

rng = np.random.default_rng(0)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'

C_LINE  = "#2C4A7C"
C_CTX   = "#5B7DB5"
C_HOR   = "#CA6F6A"
C_FDATE = "#E67E22"
C_GRAY  = "#9CA3AF"

T = 80
t = np.arange(T)
series = (
    1.5
    + 0.6 * np.sin(2 * np.pi * t / 30)
    + 0.3 * np.sin(2 * np.pi * t / 12)
    + rng.normal(0, 0.15, T)
)

CONTEXT    = 20
HORIZON    = 12
SLIDE_END  = T - CONTEXT   # = 60: slide until context end hits T (horizon truncates)

HOLD_BUILD = 1
HOLD_SLIDE = 2
PAUSE      = 6

# Phase 1: buildup — ws=0, we grows 1..CONTEXT
# Phase 2: slide   — ws grows 0..SLIDE_END, we = ws+CONTEXT (horizon truncates near end)
frame_states = []

for we in range(1, CONTEXT + 1):
    frame_states += [(0, we)] * HOLD_BUILD

for ws in range(0, SLIDE_END + 1):
    frame_states += [(ws, ws + CONTEXT)] * HOLD_SLIDE

frame_states += [(SLIDE_END, SLIDE_END + CONTEXT)] * PAUSE

frame_fcd = [min(we, T - 1) for (ws, we) in frame_states]

# Trail covers all slide positions
trail_ws_list = list(range(0, SLIDE_END + 1))

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 3.8))
fig.subplots_adjust(left=0.04, right=0.97, top=0.88, bottom=0.22)

y_min = series.min() - 0.55
y_max = series.max() + 0.82
ax.set_xlim(-1, T)
ax.set_ylim(y_min, y_max)
ax.set_xlabel("Time  t", fontsize=16)
ax.set_ylabel("y", fontsize=16, rotation=0, labelpad=16)
ax.tick_params(left=False, labelleft=False, labelsize=16)

ax.plot(t, series, color=C_LINE, linewidth=1.8, zorder=5)
ax.scatter(t, series, color=C_LINE, s=18, zorder=6)

# Rects
rect_ctx = patches.Rectangle((0, y_min), 1, y_max - y_min,
                               color=C_CTX, alpha=0.22, zorder=2)
ax.add_patch(rect_ctx)

rect_hor = patches.Rectangle((1, y_min), HORIZON, y_max - y_min,
                               color=C_HOR, alpha=0.32, zorder=1)
ax.add_patch(rect_hor)

# Vlines
vl = ax.axvline(0, color=C_CTX,  lw=1.2, ls="--", alpha=0.8, zorder=4)
vm = ax.axvline(1, color=C_GRAY, lw=1.0, ls="--", alpha=0.7, zorder=4)

# Both brackets on same top row — they never overlap (ctx ends where hor starts)
brk_y  = y_max - 0.04
brk_yt = y_max + 0.06

arr_ctx = ax.annotate("", xy=(1, brk_y), xytext=(0, brk_y),
                       arrowprops=dict(arrowstyle="<->", color=C_CTX, lw=1.3))
lbl_ctx = ax.text(0.5, brk_yt, "context  L",
                  ha="center", va="bottom", fontsize=14,
                  color=C_CTX, fontweight="bold")

arr_hor = ax.annotate("", xy=(1 + HORIZON, brk_y), xytext=(1, brk_y),
                       arrowprops=dict(arrowstyle="<->", color=C_HOR, lw=1.3))
lbl_hor = ax.text(1 + HORIZON / 2, brk_yt, "horizon  H",
                  ha="center", va="bottom", fontsize=14,
                  color=C_HOR, fontweight="bold")

# FCD dots
dot_fcd, = ax.plot([], [], "o", color=C_FDATE, ms=11,
                   mec="white", mew=1.4, zorder=9)
lbl_fcd = ax.text(0, 0, "", ha="left", va="top",
                  fontsize=14, color=C_FDATE, fontweight="bold", zorder=10)

past_dots, = ax.plot([], [], "o", color=C_FDATE, ms=8,
                     mec="white", mew=1.0, alpha=0.55, zorder=8)

# Trail rects — one per unique `we` value across BOTH buildup and slide
# buildup: we = 1..CONTEXT, slide: we = CONTEXT..CONTEXT+SLIDE_END
all_we_values = list(range(1, SLIDE_END + CONTEXT + 1))   # 1..80
trail_rects = {}   # we_val → rect
for we_val in all_we_values:
    hor_start = we_val
    hor_w     = max(0, min(HORIZON, T - hor_start))
    r = patches.Rectangle((hor_start, y_min), hor_w, y_max - y_min,
                           color=C_HOR, alpha=0.0, zorder=1)
    ax.add_patch(r)
    trail_rects[we_val] = r


def update(fi):
    ws, we = frame_states[fi]
    wh_raw = we + HORIZON
    wh     = min(wh_raw, T)          # truncated horizon end
    hor_w  = wh - we                  # actual visible horizon width

    ws_f   = float(ws)
    we_f   = float(we)
    we_i   = min(int(round(we_f)), T - 1)
    ctx_w  = we_f - ws_f

    # Rects
    rect_ctx.set_x(ws_f)
    rect_ctx.set_width(ctx_w)
    rect_hor.set_x(we_f)
    rect_hor.set_width(max(0, hor_w))

    # Vlines
    vl.set_xdata([ws_f, ws_f])
    vm.set_xdata([we_f, we_f])

    # Brackets — same row; hide labels when bracket too narrow for text
    arr_ctx.xy    = (we_f, brk_y);  arr_ctx.xyann = (ws_f, brk_y)
    lbl_ctx.set_x(ws_f + ctx_w / 2)
    lbl_ctx.set_visible(ctx_w > 8)

    if hor_w > 1:
        arr_hor.xy    = (wh, brk_y);  arr_hor.xyann = (we_f, brk_y)
        lbl_hor.set_x(we_f + hor_w / 2)
        arr_hor.set_visible(True)
        lbl_hor.set_visible(hor_w > 6)
    else:
        arr_hor.set_visible(False)
        lbl_hor.set_visible(False)

    # Current FCD
    dot_fcd.set_data([we_f], [series[we_i]])
    lbl_fcd.set_text(f"FCD  t={we_i}")
    # flip label left when near right edge
    if we_f > T - 14:
        lbl_fcd.set_ha("right")
        lbl_fcd.set_position((we_f - 0.5, series[we_i] - 0.15))
    else:
        lbl_fcd.set_ha("left")
        lbl_fcd.set_position((we_f + 0.5, series[we_i] - 0.15))

    # Accumulated past FCD dots
    past_fcd_positions = list(dict.fromkeys(frame_fcd[:fi]))
    px = [p for p in past_fcd_positions if p != we_i and p < T]
    py = [series[p] for p in px]
    past_dots.set_data(px, py)

    # Trail — reveal all past horizon zones (keyed by we value)
    for we_val, r in trail_rects.items():
        r.set_alpha(0.04 if we_val < we_i else 0.0)

    return ([rect_ctx, rect_hor, vl, vm, dot_fcd, lbl_fcd, past_dots,
             arr_ctx, lbl_ctx, arr_hor, lbl_hor] + list(trail_rects.values()))


ani = FuncAnimation(fig, update, frames=len(frame_states), interval=40, blit=False)
ani.save("forking_sequences.gif", writer=PillowWriter(fps=25), dpi=130)
print(f"✓ saved  ({len(frame_states)} frames)")